# Ray tracing spheres

*graphics · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/graphics/ray_tracing_spheres)

A tiny Whitted-style ray tracer where every pixel's ray is one row of an $(H\cdot W, 3)$ tensor. All rays are tested against all spheres at once with the quadratic formula, giving a (rays $\times$ spheres) table whose argmin is the visible surface. Lambert and Blinn–Phong shading, hard shadows from a second batch of shadow rays, mirror bounces and a checkered floor, while the light or the camera circles the scene.

**Vector form**

$$\mathbf x(t) = \mathbf o + t\,\hat{\mathbf d},\quad \lVert \mathbf x - \mathbf c_s\rVert^2 = r_s^2 \;\Rightarrow\; t = -\mathbf b \pm \sqrt{b^2 - q},\qquad I = k_a\rho + V\big[\rho\,\max(0,\hat{\mathbf n}\cdot\hat{\mathbf l}) + k_s \max(0,\hat{\mathbf n}\cdot\hat{\mathbf h})^{\alpha}\big],\quad \hat{\mathbf r} = \hat{\mathbf d} - 2(\hat{\mathbf d}\cdot\hat{\mathbf n})\,\hat{\mathbf n}$$

**Index (tensor) form**

$$b_{ps} = (o_{pi} - c_{si})\,d_{pi},\quad q_{ps} = (o-c)_{psi}(o-c)_{psi} - r_s^2,\quad t_{ps} = -b_{ps} - \sqrt{b_{ps}^2 - q_{ps}},\quad s^*_p = \arg\min_s t_{ps}\qquad r_{pi} = d_{pi} - 2\,d_{pj}n_{pj}\,n_{pi}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $o_{pi},\ d_{pi}$ | `camera_rays(eye, target, fov, W, H)` | (H·W, 3) | origin and unit direction of the ray through pixel $p$; $d$ is the screen point $(x, y, 1)$ in the camera basis, an einsum over the basis index |
| $c_{si},\ r_s$ | `sc['centres'], sc['radii']` | (S, 3), (S,) | centre and radius of sphere $s$ |
| $b_{ps}$ | `einsum('psi,pi->ps', oc, d)` | (P, S) | repeated $i$ ⇒ summed: every ray against every sphere in one call |
| $t_{ps}$ | `sphere_hits(o, d, c, r)` | (P, S) | nearer root of the quadratic; $\infty$ where $b^2 < q$ (the ray misses) or the hit is behind |
| $s^*_p = \arg\min_s t_{ps}$ | `t_all.min(dim=1)` | (P,) | the visible surface; the ground plane $y = 0$ is an extra column with $t = -o_y/d_y$ |
| $n_{pi}l_{pi},\ n_{pi}h_{pi}$ | `lambert(n, l), blinn_phong(n, l, v, α)` | (P,) | diffuse and specular factors, clamped at 0 on the far side |
| $V_p$ | `lit(p, n, light, c, r)` | (P,) | shadow rays towards the light: 1 if no sphere is hit before the light, else 0 |
| $r_{pi}$ | `reflect(d, n)` | (P, 3) | repeated $j$ ⇒ summed: mirror direction, the next batch of rays |

The colour of a pixel is $I = (1-m)\,I_\text{local} + m\,I_\text{reflected}$, where $m$ is the mirror fraction of the surface; the reflected colour comes from tracing the batch of reflected rays again, which is the recursive idea of Whitted (1980). The shading model is Phong's (1975) with Blinn's (1977) half vector $\hat{\mathbf h}$. Because $\hat{\mathbf d}$ has unit length the quadratic's leading coefficient is 1, so only $b$ and $q$ are needed. Secondary rays start a tiny distance $\varepsilon$ off the surface, otherwise rounding makes them hit their own sphere (shadow acne). Try zero bounces to see the scene without reflections, a high shininess for small sharp highlights, or a wide field of view. Real renderers add soft shadows, refraction and many samples per pixel (path tracing), but the batched ray-against-everything table stays the same.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/graphics/ray_tracing_spheres/main.py
"""Main loop: parameters → tensors → solve → plot."""

import time
from pathlib import Path

import torch

from problems._common.plotting import frame_indices
from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import sphere_hits


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    sc = tensors.scene(dev)
    target = torch.tensor([0.0, 0.7, 0.0], device=dev)
    n = params["frames"]
    if params["orbit"] == "light":  # the light circles the scene, the camera stands still
        lights = tensors.orbit(n, 4.0, 5.0, 0.8, dev)
        eyes = torch.tensor([0.0, 2.2, 7.5], device=dev).expand(n, 3)
    else:  # the camera circles the scene, the light stands still
        lights = torch.tensor([3.0, 5.0, 2.5], device=dev).expand(n, 3)
        eyes = tensors.orbit(n, 7.5, 2.2, 0.0, dev)
    keep = frame_indices(n, n)  # every frame (only a handful in fast mode)
    eyes, lights = eyes[keep], lights[keep]

    w, h = params["width"], params["height"]
    o, d = tensors.camera_rays(eyes[0], target, params["fov"], w, h)
    t = sphere_hits(o, d, sc["centres"], sc["radii"])
    print(f"{o.shape[0]} rays (one per pixel of {w}×{h}) against {len(sc['radii'])} spheres: t table {tuple(t.shape)}")

    start = time.perf_counter()
    images, stats = solve.render(sc, eyes, lights, target, params["fov"], w, h, params["bounces"], params["shininess"])
    elapsed = time.perf_counter() - start
    s = stats[0]
    print(f"first rays: {s['spheres']:.1%} hit a sphere, {s['ground']:.1%} the ground, {s['sky']:.1%} the sky")
    print(f"ground pixels in shadow (frame 0): {s['shadow']:.1%}")
    print(f"{params['bounces']} mirror bounce(s) + shadow rays: {len(images)} frames in {elapsed:.1f} s")
    return plot.animate(images, eyes, lights, sc, params["orbit"], out_dir / "ray_tracing_spheres.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/graphics/ray_tracing_spheres/equation.py
r"""Optics of a ray tracer → tensor translation.

Every pixel sends one ray x(t) = o + t d with |d| = 1. All P rays form one (P, 3) tensor and all S spheres one (S, 3)
tensor, so the intersection test is a (P, S) table computed at once. Putting the ray into |x − c_s|² = r_s² gives
a quadratic in t (the leading coefficient is d·d = 1):

    t² + 2 b_{ps} t + q_{ps} = 0,    b_{ps} = (o_{pi} − c_{si}) d_{pi},    q_{ps} = (o − c)_{psi}(o − c)_{psi} − r_s²,

    t_{ps} = −b_{ps} − \sqrt{b_{ps}² − q_{ps}}          (i summed in b and q; no real root ⇒ the ray misses).

The nearest hit is the argmin over s. At the hit point with normal n, light direction l and view direction v:

    Lambert (diffuse)   max(0, n_i l_i)
    Blinn–Phong         max(0, n_i h_i)^α,   h = (l + v)/|l + v|
    mirror reflection   r_i = d_i − 2 (d_j n_j) n_i                  (j summed)

A shadow ray from the hit point towards the light is the same intersection test: if it hits a sphere before
reaching the light, the point gets no direct light.
"""

import torch

EPS = 1e-4  # start secondary rays a little off the surface so they do not hit it again


def normalize(v: torch.Tensor) -> torch.Tensor:
    """v / |v| along the last axis."""
    return v / torch.linalg.vector_norm(v, dim=-1, keepdim=True)


def sphere_hits(o: torch.Tensor, d: torch.Tensor, c: torch.Tensor, r: torch.Tensor) -> torch.Tensor:
    r"""Distance t_{ps} from ray p to sphere s (inf if missed).   o, d: (P, 3), c: (S, 3), r: (S,) → (P, S)"""
    oc = o[:, None, :] - c[None, :, :]  # (P, S, 3)
    b = torch.einsum("psi,pi->ps", oc, d)  # i summed: b = (o − c)·d
    q = torch.einsum("psi,psi->ps", oc, oc) - r**2  # i summed: |o − c|² − r²
    disc = b**2 - q  # discriminant / 4
    root = disc.clamp(min=0).sqrt()
    t = torch.where(-b - root > EPS, -b - root, -b + root)  # near root, or far root if we start inside
    return torch.where((disc >= 0) & (t > EPS), t, torch.full_like(t, torch.inf))


def plane_hits(o: torch.Tensor, d: torch.Tensor, height: float = 0.0) -> torch.Tensor:
    r"""Distance to the ground plane y = height: t_p = (height − o_{py}) / d_{py} (inf if behind).   → (P,)"""
    t = (height - o[:, 1]) / d[:, 1]
    return torch.where(t > EPS, t, torch.full_like(t, torch.inf))


def lambert(n: torch.Tensor, light_dir: torch.Tensor) -> torch.Tensor:
    r"""Diffuse factor max(0, n_i l_i)   (i summed; l = light_dir).   (P, 3), (P, 3) → (P,)"""
    return torch.einsum("pi,pi->p", n, light_dir).clamp(min=0)


def blinn_phong(n: torch.Tensor, light_dir: torch.Tensor, v: torch.Tensor, shininess: float) -> torch.Tensor:
    r"""Specular factor max(0, n_i h_i)^α with the half vector h = (l + v)/|l + v|.   → (P,)"""
    h = normalize(light_dir + v)
    return torch.einsum("pi,pi->p", n, h).clamp(min=0) ** shininess


def reflect(d: torch.Tensor, n: torch.Tensor) -> torch.Tensor:
    r"""Mirror direction r_i = d_i − 2 (d_j n_j) n_i   (j summed).   (P, 3) → (P, 3)"""
    return d - 2 * torch.einsum("pj,pj->p", d, n)[:, None] * n


def lit(p: torch.Tensor, n: torch.Tensor, light: torch.Tensor, c: torch.Tensor, r: torch.Tensor) -> torch.Tensor:
    r"""Shadow rays: True where nothing blocks the segment from p to the light.   p, n: (P, 3), light: (3,) → (P,)"""
    to_light = light - p  # (P, 3)
    dist = torch.linalg.vector_norm(to_light, dim=-1)  # (P,)
    t = sphere_hits(p + EPS * n, to_light / dist[:, None], c, r)  # (P, S), a second batch of rays
    return t.min(dim=1).values > dist  # the ground cannot block a light above it, so only spheres are tested


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/graphics/ray_tracing_spheres/tensors.py
"""Build the concrete tensors: the scene (spheres + ground plane), the camera rays and the orbits."""

import math

import torch

from .equation import normalize


def scene(device: torch.device) -> dict[str, torch.Tensor]:
    """S spheres: centres c_{si} (S, 3), radii r_s (S,), colours (S, 3) and mirror fractions (S,)."""
    centres = torch.tensor(
        [[0.0, 1.0, 0.0], [-2.1, 0.6, 0.9], [1.9, 0.7, 1.0], [0.7, 0.35, 2.1], [-1.0, 0.45, -1.9]], device=device
    )
    radii = torch.tensor([1.0, 0.6, 0.7, 0.35, 0.45], device=device)
    albedo = torch.tensor(
        [[0.85, 0.85, 0.88], [0.86, 0.15, 0.15], [0.15, 0.39, 0.92], [0.09, 0.64, 0.29], [0.96, 0.62, 0.04]],
        device=device,
    )
    mirror = torch.tensor([0.7, 0.1, 0.25, 0.05, 0.15], device=device)  # 0 = matte, 1 = perfect mirror
    return {"centres": centres, "radii": radii, "albedo": albedo, "mirror": mirror}


def camera_basis(eye: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """Rows: right, up, forward unit vectors of a camera at eye looking at target (no roll): (3, 3)."""
    forward = normalize(target - eye)
    right = normalize(torch.linalg.cross(forward, torch.tensor([0.0, 1.0, 0.0], device=eye.device)))
    up = torch.linalg.cross(right, forward)
    return torch.stack([right, up, forward])


def camera_rays(
    eye: torch.Tensor, target: torch.Tensor, fov_deg: float, width: int, height: int
) -> tuple[torch.Tensor, torch.Tensor]:
    """One ray per pixel: origins o_{pi} and unit directions d_{pi}, both (H·W, 3), row p = i·W + j."""
    dev = eye.device
    half = math.tan(math.radians(fov_deg) / 2)  # half-height of the image plane at distance 1
    xs = ((torch.arange(width, device=dev) + 0.5) / width * 2 - 1) * half * width / height  # (W,)
    ys = (1 - (torch.arange(height, device=dev) + 0.5) / height * 2) * half  # (H,) image rows go down
    y, x = torch.meshgrid(ys, xs, indexing="ij")  # (H, W)
    screen = torch.stack([x, y, torch.ones_like(x)], dim=-1).reshape(-1, 3)  # (P, 3) = (right, up, forward) weights
    d = normalize(torch.einsum("pk,ki->pi", screen, camera_basis(eye, target)))  # k summed over the 3 basis vectors
    return eye.expand_as(d), d


def orbit(n_frames: int, radius: float, height: float, phase: float, device: torch.device) -> torch.Tensor:
    """Positions on a horizontal circle, one per frame: (F, 3)."""
    angle = phase + torch.linspace(0, 2 * math.pi, n_frames + 1, device=device)[:-1]
    return torch.stack([radius * torch.sin(angle), torch.full_like(angle, height), radius * torch.cos(angle)], dim=-1)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/graphics/ray_tracing_spheres/solve.py
"""Trace every pixel's ray through the scene: nearest hit, shading, shadows and mirror bounces."""

import torch

from problems._common.progress import steps

from . import tensors
from .equation import EPS, blinn_phong, lambert, lit, normalize, plane_hits, reflect, sphere_hits

AMBIENT = 0.08  # light that arrives from everywhere, so shadows are not pitch black
SPECULAR = 0.6  # strength of the Blinn–Phong highlight
GROUND_MIRROR = 0.15  # the floor reflects a little too


def sky(d: torch.Tensor) -> torch.Tensor:
    """Background colour for rays that hit nothing: white at the horizon, blue overhead.   (P, 3) → (P, 3)"""
    a = d[:, 1:2].clamp(0, 1)
    return (1 - a) * torch.tensor([0.95, 0.96, 0.98], device=d.device) + a * torch.tensor(
        [0.45, 0.65, 0.95], device=d.device
    )


def checker(p: torch.Tensor) -> torch.Tensor:
    """Checkerboard colour of the ground at points p: 1 × 1 squares, white and grey.   (P, 3) → (P, 3)"""
    parity = (torch.floor(p[:, 0]) + torch.floor(p[:, 2])).remainder(2)[:, None]  # (P, 1), 0 or 1
    return 0.9 - 0.45 * parity.expand(-1, 3)


def first_hit(o: torch.Tensor, d: torch.Tensor, sc: dict[str, torch.Tensor]):
    """Nearest surface of every ray: distance t (P,), index k (P,) with k = S for the ground, hit mask (P,)."""
    t_all = torch.cat([sphere_hits(o, d, sc["centres"], sc["radii"]), plane_hits(o, d)[:, None]], dim=1)  # (P, S+1)
    t, k = t_all.min(dim=1)  # argmin over the S spheres and the plane
    return t, k, torch.isfinite(t)


def surface(p: torch.Tensor, k: torch.Tensor, sc: dict[str, torch.Tensor]):
    """Normal n (P, 3), colour (P, 3) and mirror fraction (P,) at hit points p on surface k."""
    n_spheres = len(sc["radii"])
    on_ground = k == n_spheres
    s = k.clamp(max=n_spheres - 1)  # a valid sphere index for every ray (ground rays are overwritten below)
    n = (p - sc["centres"][s]) / sc["radii"][s][:, None]
    n = torch.where(on_ground[:, None], torch.tensor([0.0, 1.0, 0.0], device=p.device), n)
    colour = torch.where(on_ground[:, None], checker(p), sc["albedo"][s])
    mirror = torch.where(on_ground, torch.full_like(sc["mirror"][s], GROUND_MIRROR), sc["mirror"][s])
    return n, colour, mirror


def trace(
    o: torch.Tensor, d: torch.Tensor, sc: dict[str, torch.Tensor], light: torch.Tensor, bounces: int, shininess: float
) -> tuple[torch.Tensor, dict[str, float]]:
    """Colour of every ray (P, 3) after up to `bounces` mirror reflections, plus a few statistics of the first hit."""
    colour = torch.zeros_like(d)
    weight = torch.ones(d.shape[0], 1, device=d.device)  # how much of the pixel the current ray still carries
    stats = {}
    for bounce in range(bounces + 1):
        t, k, hit = first_hit(o, d, sc)
        p = o + torch.where(hit, t, torch.zeros_like(t))[:, None] * d  # misses stay at o (they get weight 0 below)
        n, albedo, mirror = surface(p, k, sc)
        light_dir = normalize(light - p)  # (P, 3) unit vector l towards the light
        v = -d  # towards the viewer
        visible = lit(p, n, light, sc["centres"], sc["radii"]).float()[:, None]  # (P, 1): 1 lit, 0 in shadow
        direct = visible * (
            lambert(n, light_dir)[:, None] * albedo + SPECULAR * blinn_phong(n, light_dir, v, shininess)[:, None]
        )
        local = AMBIENT * albedo + direct  # (P, 3)

        m = mirror[:, None] if bounce < bounces else torch.zeros_like(mirror[:, None])  # last bounce: no mirror
        colour += weight * torch.where(hit[:, None], (1 - m) * local, sky(d))
        weight = weight * torch.where(hit[:, None], m, torch.zeros_like(m))
        if bounce == 0:
            ground = hit & (k == len(sc["radii"]))
            stats = {
                "spheres": (hit & ~(k == len(sc["radii"]))).float().mean().item(),
                "ground": ground.float().mean().item(),
                "sky": (~hit).float().mean().item(),
                "shadow": (1 - visible[ground, 0]).mean().item() if ground.any() else 0.0,
            }
        # The next batch: mirror rays from every hit point (rays that escaped to the sky just carry on, unweighted).
        o = torch.where(hit[:, None], p + EPS * n, o)
        d = torch.where(hit[:, None], reflect(d, n), d)
    return colour.clamp(0, 1), stats


def render(
    sc: dict[str, torch.Tensor],
    eyes: torch.Tensor,
    lights: torch.Tensor,
    target: torch.Tensor,
    fov: float,
    width: int,
    height: int,
    bounces: int,
    shininess: float,
) -> tuple[torch.Tensor, list[dict[str, float]]]:
    """Images for every frame: eyes, lights (F, 3) → (F, H, W, 3) and first-hit statistics per frame."""
    frames, stats = [], []
    for f in steps(len(eyes), "frames"):
        o, d = tensors.camera_rays(eyes[f], target, fov, width, height)
        rgb, s = trace(o, d, sc, lights[f], bounces, shininess)
        frames.append(rgb.reshape(height, width, 3))
        stats.append(s)
    return torch.stack(frames), stats


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/graphics/ray_tracing_spheres/plot.py
"""Animate the rendered image next to a top-down map of the scene, the camera and the light."""

import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Circle

from problems._common.plotting import save_gif


def animate(images: torch.Tensor, eyes: torch.Tensor, lights: torch.Tensor, sc: dict, orbit: str, path: Path) -> Path:
    """images (F, H, W, 3) in [0, 1], eyes and lights (F, 3)."""
    img = (images.cpu().numpy() * 255).astype(np.uint8)
    eye, light = eyes.cpu().numpy(), lights.cpu().numpy()

    fig, (ax_i, ax_m) = plt.subplots(1, 2, figsize=(8.4, 4), gridspec_kw={"width_ratios": [1.6, 1]})
    image = ax_i.imshow(img[0], interpolation="bilinear")
    ax_i.set_xticks([])
    ax_i.set_yticks([])
    ax_i.grid(False)
    title = ax_i.set_title("", fontsize=10)

    for c, r, a in zip(sc["centres"].cpu().numpy(), sc["radii"].cpu().numpy(), sc["albedo"].cpu().numpy()):
        ax_m.add_patch(Circle((c[0], c[2]), r, facecolor=a, edgecolor="#374151", lw=1))
    angle = np.linspace(0, 2 * math.pi, 100)
    moving = light if orbit == "light" else eye
    radius = np.hypot(moving[0, 0], moving[0, 2])
    ax_m.plot(radius * np.sin(angle), radius * np.cos(angle), "--", color="#9ca3af", lw=1)
    (cam,) = ax_m.plot([], [], "s", color="#374151", ms=8, label="camera")
    (sight,) = ax_m.plot([], [], "-", color="#374151", lw=1)
    (lamp,) = ax_m.plot([], [], "*", color="#f59e0b", ms=15, mec="#b45309", label="light")
    ax_m.set_xlim(-8.5, 8.5)
    ax_m.set_ylim(8.5, -8.5)  # +z points towards the viewer at the bottom
    ax_m.set_aspect("equal")
    ax_m.set_xlabel("x")
    ax_m.set_ylabel("z")
    ax_m.set_title("top view", fontsize=10)
    ax_m.legend(loc="upper right", fontsize=8)

    def update(f: int):
        image.set_data(img[f])
        cam.set_data([eye[f, 0]], [eye[f, 2]])
        sight.set_data([eye[f, 0], 0], [eye[f, 2], 0])
        lamp.set_data([light[f, 0]], [light[f, 2]])
        title.set_text(f"ray traced, {orbit} orbiting — frame {f + 1}/{len(img)}")

    fig.tight_layout()
    return save_gif(fig, update, list(range(len(img))), path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'orbit': 'light',
    'bounces': 2,
    'shininess': 60.0,
    'fov': 45.0,
    'width': 200,
    'height': 140,
    'frames': 48
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.graphics.ray_tracing_spheres")]:
    del sys.modules[name]
main = importlib.import_module("problems.graphics.ray_tracing_spheres.main")
problem_dir = Path("problems/graphics/ray_tracing_spheres")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
